# Introduction
This notebook is used to fetch, plot, and analyze experiment results.

## 1. Initial Setup

In [44]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3896


In [45]:
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Fetch Results

In [46]:
import seml
import pandas as pd

db_collection = 'llama-pert-awq-bnb-hqq'
states = ["COMPLETED"]

# Get the results
all_results = seml.evaluation.get_results(db_collection, to_data_frame=True, states=states)

print(f"Length of all_results before deduplication: {len(all_results)}")

# Get the list of columns that start with 'config.'
config_columns = [col for col in all_results.columns if col.startswith('config.')]

# Drop duplicates based on config columns, keeping the last occurrence
all_results = all_results.drop_duplicates(subset=config_columns, keep='last')

print(f"Length of all_results after deduplication: {len(all_results)}")
print("Columns used for deduplication:")
print(config_columns)
print("\nAll columns in the dataframe:")
print(all_results.columns)
all_results.head()

Output()

Output()

Length of all_results before deduplication: 4800
Length of all_results after deduplication: 4800
Columns used for deduplication:
['config.overwrite', 'config.db_collection', 'config.batch_size', 'config.dataset_name', 'config.device', 'config.exp_id', 'config.max_entries', 'config.max_new_tokens', 'config.model_name', 'config.n_beams', 'config.n_repeats', 'config.num_excel_rows', 'config.save_excel', 'config.seed', 'config.strategy', 'config.temperature', 'config.typo_intensity', 'config.typo_type', 'config.use_beam_search']

All columns in the dataframe:
Index(['_id', 'config.overwrite', 'config.db_collection', 'config.batch_size',
       'config.dataset_name', 'config.device', 'config.exp_id',
       'config.max_entries', 'config.max_new_tokens', 'config.model_name',
       'config.n_beams', 'config.n_repeats', 'config.num_excel_rows',
       'config.save_excel', 'config.seed', 'config.strategy',
       'config.temperature', 'config.typo_intensity', 'config.typo_type',
       'config

,_id,config.overwrite,config.db_collection,config.batch_size,config.dataset_name,config.device,config.exp_id,config.max_entries,config.max_new_tokens,config.model_name,...,result.Brier_adj,result.LogLoss_adj,result.Entropy_adj,result.AUCROC_sem,result.AUCPR_sem,result.Brier_sem,result.LogLoss_sem,result.Entropy_sem,result.Accuracy,result.fail_trace
0,1,1,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.561755,5.918077,36.896553,1.0,1.0,0.0,2.220446e-16,-3.981839e-08,0.396552,<function get_results at 0x7f842c07c280>
1,2,2,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.561755,5.918077,36.896553,1.0,1.0,0.0,2.220446e-16,-3.981839e-08,0.396552,<function get_results at 0x7f842c07c280>
2,3,3,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.561755,5.918077,36.896553,1.0,1.0,0.0,2.220446e-16,-3.981839e-08,0.396552,<function get_results at 0x7f842c07c280>
3,4,4,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.579266,5.823508,45.919863,1.0,1.0,0.0,2.220446e-16,-3.765434e-08,0.375000,<function get_results at 0x7f842c07c280>
4,5,5,llama-pert-awq-bnb-hqq,32,P101,cuda,pert-awq-bnb-hqq-10-07,None,25,Llama-3-8B,...,0.624608,6.945512,41.225135,1.0,1.0,0.0,2.220446e-16,-3.246064e-08,0.323276,<function get_results at 0x7f842c07c280>


### 2.1 Check Failed Rows

In [8]:
import seml
import pandas as pd
from collections import Counter

db_collection = 'llama-pert-awq-bnb-hqq'
states = ["FAILED"]

# Get the results
failed_results = seml.evaluation.get_results(db_collection, to_data_frame=True, states=states)
print(f"Length of failed_results before deduplication: {len(failed_results)}")

# Get the list of columns that start with 'config.'
config_columns = [col for col in failed_results.columns if col.startswith('config.')]

# Drop duplicates based on config columns, keeping the last occurrence
failed_results = failed_results.drop_duplicates(subset=config_columns, keep='last')
print(f"Length of failed_results after deduplication: {len(failed_results)}")

print("\nUnique values and their frequencies for each config parameter in failed_results:")
for col in ["config.dataset_name", "config.model_name", "config.typo_type", "config.typo_intensity"]:
    value_counts = Counter(failed_results[col])
    print(f"\n{col}:")
    for value, count in value_counts.items():
        print(f"  - {value}: {count}")

Output()

Output()

Length of failed_results before deduplication: 70
Length of failed_results after deduplication: 70

Unique values and their frequencies for each config parameter in failed_results:

config.dataset_name:
  - P364: 11
  - P37: 18
  - P740: 21
  - P101: 20

config.model_name:
  - Llama-3-8B-BNB-4bit-local: 50
  - Llama-3-8B-HQQ-mixed-local: 20

config.typo_type:
  - word_phrase_translation: 2
  - word_context_aware_insertion: 1
  - word_remove_punctuation: 5
  - word_keyword_only: 5
  - word_taxonomy_pos: 6
  - word_taxonomy_neg: 5
  - none: 5
  - char_insertion: 5
  - char_deletion: 6
  - char_replacement: 4
  - char_repetition: 5
  - char_swapping: 3
  - word_CMW: 4
  - char_LCC: 5
  - word_synonym: 1
  - char_insert_noise: 3
  - word_repeat: 1
  - char_substitution: 3
  - word_emoji: 1

config.typo_intensity:
  - 1: 24
  - 2: 23
  - 3: 23


### 2.2 Check high accuracy columns

In [9]:
# Filter rows where accuracy is higher than 0.6
high_accuracy_results = all_results[all_results['result.Accuracy'] > 0.6]

# Print the number of rows that meet this criteria
print(f"Number of rows with accuracy > 0.6: {len(high_accuracy_results)}")

# Display the first few rows of the filtered results
print(high_accuracy_results[['config.model_name', 'config.strategy', 'result.Accuracy']].head())

# Count the number of high accuracy rows for each unique model name
model_counts = high_accuracy_results['config.model_name'].value_counts()

print("\nNumber of high accuracy rows for each model:")
print(model_counts)

# Optional: Calculate and print the percentage of high accuracy rows for each model
total_rows = len(all_results)
model_percentages = (model_counts / total_rows * 100).round(2)

print("\nPercentage of high accuracy rows for each model:")
print(model_percentages)

Number of rows with accuracy > 0.6: 2656
   config.model_name    config.strategy  result.Accuracy
54        Llama-3-8B  Direct Completion         0.669540
55        Llama-3-8B  Direct Completion         0.744253
56        Llama-3-8B  Direct Completion         0.778736
60        Llama-3-8B  Direct Completion         0.939611
61        Llama-3-8B  Direct Completion         0.939611

Number of high accuracy rows for each model:
config.model_name
Llama-3-8B                    756
Llama-3-8B-AWQ-4bit-local     714
Llama-3-8B-BNB-4bit-local     620
Llama-3-8B-HQQ-mixed-local    566
Name: count, dtype: int64

Percentage of high accuracy rows for each model:
config.model_name
Llama-3-8B                    15.75
Llama-3-8B-AWQ-4bit-local     14.88
Llama-3-8B-BNB-4bit-local     12.92
Llama-3-8B-HQQ-mixed-local    11.79
Name: count, dtype: float64


## 3. Plot results

In [69]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from fpdf import FPDF
import numpy as np
from PIL import Image

# Add these imports at the top
import matplotlib.pyplot as plt

# import matplotlib
# matplotlib.rcParams['text.usetex'] = True

# Replace with:
import matplotlib
matplotlib.rcParams['text.usetex'] = False

# If you still see font-related issues, you might also want to add:
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

from src.models import get_model_num_bits

# Define color scheme for the four models
model_colors = {
    'Llama-3-8B': '#1f77b4',  # Blue
    'Llama-3-8B-AWQ-4bit-local': '#ff7f0e',  # Orange
    'Llama-3-8B-BNB-4bit-local': '#2ca02c',  # Green
    'Llama-3-8B-HQQ-mixed-local': '#d62728'  # Red
}

# Define color scheme for intensities
intensity_colors = {
    1: '#1f77b4',  # Blue
    2: '#2ca02c',  # Green
    3: '#d62728'   # Red
}

model_name_map = {
    'Llama-3-8B': 'Llama-3-8B',
    'Llama-3-8B-AWQ-4bit-local': 'Llama-3-8B-AWQ-4bit',
    'Llama-3-8B-BNB-4bit-local': 'Llama-3-8B-BNB-4bit',
    'Llama-3-8B-HQQ-mixed-local': 'Llama-3-8B-HQQ-3-4bit'
}

## 3.1 Radar plots

In [70]:
def create_comparison_radar_plots(df, metric, intensity):
    # Create subplots with reduced size
    fig = make_subplots(rows=3, cols=1,
                       specs=[[{'type': 'polar'}]]*3,
                       subplot_titles=[
                           "Llama vs BNB",
                           "Llama vs AWQ",
                           "Llama vs HQQ"
                       ],
                       vertical_spacing=0.15)
    
    df_intensity = df[df['config.typo_intensity'] == intensity]
    base_model = 'Llama-3-8B'
    comparison_models = ['Llama-3-8B-BNB-4bit-local', 'Llama-3-8B-AWQ-4bit-local', 'Llama-3-8B-HQQ-mixed-local']
    
    # Get baseline value
    baseline_value = df[(df['config.typo_type'] == 'none') &
                       (df['config.model_name'] == base_model)][f'result.{metric}'].mean()
    
    for i, comp_model in enumerate(comparison_models):
        shown_models = [base_model, comp_model]  # Track models for this subplot
        for model in shown_models:
            df_model = df_intensity[df_intensity['config.model_name'] == model]
            if df_model.empty:
                print(f"Warning: No data for model {model} with intensity {intensity} and metric {metric}")
                continue
                
            df_avg = df_model[df_model['config.typo_type'] != 'none'].groupby('config.typo_type')[f'result.{metric}'].mean().reset_index()
            values = df_avg[f'result.{metric}'].tolist()
            if not values:
                print(f"Warning: No values for model {model} with intensity {intensity} and metric {metric}")
                continue
                
            theta = df_avg['config.typo_type'].apply(lambda x: x.replace('_', ' ')).tolist()
            
            values.append(values[0])
            theta.append(theta[0])
            
            # Show legend only for current subplot's models
            show_legend = model in [base_model, comp_model]
            
            fig.add_trace(
                go.Scatterpolar(
                    r=values,
                    theta=theta,
                    fill='toself',
                    name=model_name_map[model],
                    line=dict(color=model_colors[model]),
                    showlegend=show_legend,
                    legendgroup=f'group_{i}'  # Group legends by subplot
                ),
                row=i+1, col=1
            )
            
            # Add baseline circle only for current subplot
            if model == comp_model:  # Add baseline only once per subplot
                fig.add_trace(
                    go.Scatterpolar(
                        r=[baseline_value] * len(theta),
                        theta=theta,
                        name='Baseline',
                        line=dict(color='gray', dash='dash'),
                        showlegend=True,
                        legendgroup=f'group_{i}'
                    ),
                    row=i+1, col=1
                )
    
    # Update layout with reduced size and improved formatting
    all_model_names = [model_name_map[m] for m in [base_model] + comparison_models]
    fig.update_layout(
        height=500,  # Further reduced
        width=400,   # Further reduced
        title=dict(
            text=f'Model Comparison: Radar Plots for {", ".join(all_model_names)} Accuracy',
            font=dict(size=16)
        ),
        font=dict(size=16),  # Increased legend font size
        showlegend=True,
        margin=dict(t=120, b=50),
        annotations=[
            dict(
                text='sum(I(true_answer in generated_answer)) / len(dataset)',
                showarrow=False,
                xref='paper',
                yref='paper',
                x=0,
                y=-0.1,
                font=dict(size=12)
            )
        ]
    )
    
    # Update subplot titles with larger font
    for i in range(len(fig.layout.annotations)):
        fig.layout.annotations[i].update(font=dict(size=16))
    
    # Update each polar subplot
    for i in range(1, 4):
        fig.update_layout(**{
            f'polar{i}': dict(
                radialaxis=dict(
                    visible=True,
                    range=[0, 1],
                    tickformat='.2f',
                    tickfont=dict(size=10),
                    gridwidth=1.5
                ),
                angularaxis=dict(
                    tickfont=dict(size=10),
                    rotation=90,
                    direction="clockwise",
                    period=len(theta) - 1,
                    tickmode='array',
                    ticktext=theta[:-1],
                    tickvals=np.linspace(0, 360, len(theta))[:-1]
                ),
                sector=[0, 360],
                domain={'y': [0.1, 0.9]}
            )
        })
    
    return fig

## 3.2 Box plots

In [71]:
def create_boxplot_comparison(df, metric):
    fig = make_subplots(rows=3, cols=1,
                       subplot_titles=[f"Intensity {i}" for i in [1, 2, 3]],
                       vertical_spacing=0.1)
    
    models = df['config.model_name'].unique()
    perturbation_types = df['config.typo_type'].unique()
    
    for intensity in [1, 2, 3]:
        row = intensity
        
        for i, model in enumerate(models):
            df_model = df[(df['config.model_name'] == model) & 
                         (df['config.typo_intensity'] == intensity)]
            
            y_data = []
            x_data = []
            
            for pert in perturbation_types:
                values = df_model[df_model['config.typo_type'] == pert][f'result.{metric}']
                y_data.extend(values)
                x_data.extend([pert] * len(values))
            
            fig.add_trace(
                go.Box(
                    y=y_data,
                    x=x_data,
                    name=f'{model_name_map[model]}',
                    marker_color=model_colors[model],
                    showlegend=(row == 1)  # Show legend only for first row
                ),
                row=row, col=1
            )
    
    # Update layout
    fig.update_layout(
        height=800,
        width=1200,
        title=dict(
            text=f'Distribution of {metric} Across Perturbation Types',
            font=dict(size=16)
        ),
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1,
            font=dict(size=14)
        ),
        font=dict(size=12)
    )
    
    # Update y-axes
    for i in range(1, 4):
        fig.update_yaxes(title_text=metric, row=i, col=1)
    
    # Update subplot titles with larger font
    for i in range(len(fig.layout.annotations)):
        fig.layout.annotations[i].update(font=dict(size=14))
    
    return fig

## 3.3 Bar plots

In [72]:
def create_all_perturbations_bar_plot(df, metric, intensity):
    fig = go.Figure()
    
    # Get all unique models
    models = df['config.model_name'].unique()
    
    # Calculate the baseline values for each model
    baseline_values = {
        model: df[(df['config.typo_type'] == 'none') & 
                 (df['config.model_name'] == model)][f'result.{metric}'].mean()
        for model in models
    }
    
    # Get perturbation types excluding 'none'
    perturbation_types = [p for p in df['config.typo_type'].unique() if p != 'none']
    
    # Create bars for each model
    for i, model in enumerate(models):
        y = []
        for pert_type in perturbation_types:
            value = df[(df['config.typo_type'] == pert_type) & 
                      (df['config.typo_intensity'] == intensity) & 
                      (df['config.model_name'] == model)][f'result.{metric}'].mean()
            y.append(value)
        
        # Add horizontal bars
        fig.add_trace(go.Bar(
            y=perturbation_types,
            x=y,
            name=f'{model_name_map[model]} ({get_model_num_bits(model)}-bit)',
            orientation='h',
            marker_color=model_colors[model]
        ))
        
        # Add baseline line for each model
        fig.add_shape(
            type="line",
            y0=-0.5,
            y1=len(perturbation_types)-0.5,
            x0=baseline_values[model],
            x1=baseline_values[model],
            line=dict(
                color=model_colors[model],
                width=2,
                dash="dash"
            )
        )
    
    # Update layout
    fig.update_layout(
        title=dict(
            text=f'All Perturbations Bar Plot of {metric} (Intensity {intensity})',
            font=dict(size=16)
        ),
        yaxis_title="Perturbation Type",
        xaxis_title=metric,
        barmode='group',
        height=400,
        width=1200,
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1,
            font=dict(size=14)
        ),
        font=dict(size=12)
    )
    
    fig.update_yaxes(autorange="reversed")
    return fig

## 3.4 PDF Visualization

In [74]:
from fpdf import FPDF
import matplotlib.pyplot as plt
import os
from PIL import Image
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

class MultipleVisualizationPDFGenerator:
    def __init__(self, plots_dir, exp_id, model_colors, model_name_map, get_model_num_bits):
        self.plots_dir = plots_dir
        self.exp_id = exp_id
        self.model_colors = model_colors
        self.model_name_map = model_name_map
        self.get_model_num_bits = get_model_num_bits
        
    class CustomPDF(FPDF):
        def __init__(self, metric):
            super().__init__()
            self.metric = metric
            
        def footer(self):
            self.set_y(-15)
            self.set_font('Helvetica', 'I', 8)
            self.cell(0, 10, f'Page {self.page_no()}', 0, 0, 'C')
            
            if self.page_no() > 1:  # Skip title page
                if self.metric in ["Accuracy", "AUCPR_sample"]:
                    plt.figure(figsize=(6, 1))
                    plt.axis('off')
                    if self.metric == "Accuracy":
                        plt.text(0.5, 0.5, 'sum(I(answer in true_answer)) / len(dataset)', 
                                fontsize=12, ha='center')
                    else:
                        plt.text(0.5, 0.5, r'$\mathrm{AUCPR} = \int_0^1 \frac{TP}{TP + FP} d(\frac{TP}{TP + FN})$', 
                                fontsize=12, ha='center')
                    
                    eq_file = f'temp_equation_{self.page_no()}.png'
                    plt.savefig(eq_file, bbox_inches='tight', dpi=300, transparent=True)
                    plt.close()
                    
                    self.image(eq_file, x=60, y=270, w=90)
                    os.remove(eq_file)
    
    def generate_config_pages(self, pdf, all_results, plot_type):
        # Add title page
        pdf.add_page()
        pdf.set_font("Helvetica", 'B', size=14)
        pdf.cell(0, 10, f"Model Comparison: {plot_type} Analysis for {pdf.metric}", ln=True, align='C')
        
        # Add configuration page
        pdf.add_page()
        pdf.set_font("Helvetica", 'B', size=14)
        pdf.cell(0, 10, "Configuration Details", ln=True)
        pdf.set_font("Helvetica", size=10)
        
        config_columns = [
            "batch_size", "dataset_name", "exp_id", "max_entries",
            "max_new_tokens", "model_name", "n_beams", "n_repeats",
            "strategy", "temperature", "typo_intensity", "typo_type",
            "use_beam_search"
        ]
        
        # Set left margin explicitly
        pdf.set_left_margin(10)
        
        for col in config_columns:
            unique_values = all_results[f"config.{col}"].unique()
            text = f"{col}: {', '.join(map(str, unique_values))}"
            
            # Reset x position before each line
            pdf.set_x(10)
            pdf.multi_cell(0, 5, text)

    def create_pdf_for_plots(self, plots, metric, plot_type, all_results):
        pdf = self.CustomPDF(metric)
        pdf.set_auto_page_break(auto=True, margin=25)
        
        self.generate_config_pages(pdf, all_results, plot_type)
        
        for plot_file, desc in plots:
            pdf.add_page()
            pdf.set_font("Helvetica", 'B', size=16)  # Increased font size for descriptions
            pdf.cell(0, 20, desc, ln=True, align='C')
            
            with Image.open(plot_file) as img:
                img_width, img_height = img.size
            
            scale_factor = (pdf.w - 20) / img_width
            scaled_height = img_height * scale_factor
            
            pdf.image(plot_file, x=10, y=pdf.get_y(), w=pdf.w-20, h=scaled_height)
        
        output_path = os.path.join(
            self.plots_dir, 
            f"model_comparison_{metric}_{plot_type}_{self.exp_id}.pdf"
        )
        pdf.output(output_path)
        print(f"Generated PDF: {output_path}")

    # Rest of the methods remain unchanged
    def generate_radar_plots(self, all_results, metric):
        plots = []
        for intensity in [1, 2, 3]:
            fig = create_comparison_radar_plots(all_results, metric, intensity)
            plot_file = os.path.join(self.plots_dir, f"radar_plot_{metric}_intensity_{intensity}.png")
            fig.write_image(plot_file)
            plots.append((plot_file, f"Radar Plot - Intensity {intensity}"))
        return [("radar", metric, plots)]

    def generate_box_plots(self, all_results, metric):
        plots = []
        fig = create_boxplot_comparison(all_results, metric)
        plot_file = os.path.join(self.plots_dir, f"box_plot_{metric}.png")
        fig.write_image(plot_file)
        plots.append((plot_file, f"Box Plot Distribution"))
        return [("boxplot", metric, plots)]

    def generate_bar_plots(self, all_results, metric):
        plots = []
        for intensity in [1, 2, 3]:
            fig = create_all_perturbations_bar_plot(all_results, metric, intensity)
            plot_file = os.path.join(self.plots_dir, f"bar_plot_{metric}_intensity_{intensity}.png")
            fig.write_image(plot_file)
            plots.append((plot_file, f"Bar Plot - Intensity {intensity}"))
        return [("barplot", metric, plots)]
    
    def generate_all_pdfs(self, all_results, metrics):
        """
        Generate PDFs for all visualization types for each metric
        
        Parameters:
        all_results (DataFrame): The complete results DataFrame
        metrics (list): List of metrics to generate visualizations for
        """
        for metric in metrics:
            # Generate all plot types
            radar_plots = self.generate_radar_plots(all_results, metric)
            box_plots = self.generate_box_plots(all_results, metric)
            bar_plots = self.generate_bar_plots(all_results, metric)
            
            # Combine all plots
            all_plot_data = radar_plots + box_plots + bar_plots
            
            # Create PDFs for each plot type
            for plot_type, metric_name, plots in all_plot_data:
                self.create_pdf_for_plots(plots, metric_name, plot_type, all_results)


def main(all_results, exp_id, model_colors, model_name_map, get_model_num_bits):
    # Create plots directory
    plots_dir = f"plots/model_comparison_{exp_id}"
    os.makedirs(plots_dir, exist_ok=True)
    
    # Initialize PDF generator
    pdf_generator = MultipleVisualizationPDFGenerator(
        plots_dir, 
        exp_id, 
        model_colors, 
        model_name_map, 
        get_model_num_bits
    )
    
    # Generate PDFs for different metrics
    metrics = ["Accuracy", "AUCPR_sample"]  # Add other metrics as needed
    pdf_generator.generate_all_pdfs(all_results, metrics)


# Sample execution:
if __name__ == "__main__":
    # Specify the experiment ID
    exp_id = "awq_hqq_bnb_comparison-10-23"
    
    # Call the main function with your dataframe and required parameters
    main(all_results, exp_id, model_colors, model_name_map, get_model_num_bits)

/tmp/ipykernel_341002/3099885922.py:49: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.

/tmp/ipykernel_341002/3099885922.py:25: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=0 use new_x=XPos.RIGHT, new_y=YPos.TOP.

/tmp/ipykernel_341002/3099885922.py:54: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.

/tmp/ipykernel_341002/3099885922.py:84: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.



Generated PDF: plots/model_comparison_awq_hqq_bnb_comparison-10-23/model_comparison_Accuracy_radar_awq_hqq_bnb_comparison-10-23.pdf
Generated PDF: plots/model_comparison_awq_hqq_bnb_comparison-10-23/model_comparison_Accuracy_boxplot_awq_hqq_bnb_comparison-10-23.pdf
Generated PDF: plots/model_comparison_awq_hqq_bnb_comparison-10-23/model_comparison_Accuracy_barplot_awq_hqq_bnb_comparison-10-23.pdf


/tmp/ipykernel_341002/3099885922.py:49: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.

/tmp/ipykernel_341002/3099885922.py:25: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=0 use new_x=XPos.RIGHT, new_y=YPos.TOP.

/tmp/ipykernel_341002/3099885922.py:54: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.

/tmp/ipykernel_341002/3099885922.py:84: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.



Generated PDF: plots/model_comparison_awq_hqq_bnb_comparison-10-23/model_comparison_AUCPR_sample_radar_awq_hqq_bnb_comparison-10-23.pdf
Generated PDF: plots/model_comparison_awq_hqq_bnb_comparison-10-23/model_comparison_AUCPR_sample_boxplot_awq_hqq_bnb_comparison-10-23.pdf
Generated PDF: plots/model_comparison_awq_hqq_bnb_comparison-10-23/model_comparison_AUCPR_sample_barplot_awq_hqq_bnb_comparison-10-23.pdf


## 3.5 Latex-based Generation

In [75]:
import os
from pylatex import Document, Section, Figure, NoEscape, Math
from pylatex.utils import italic
from pylatex.base_classes import Environment
from pylatex.package import Package
import numpy as np

class CustomDocument(Document):
    def __init__(self):
        super().__init__()
        self.packages.append(Package('graphicx'))
        self.packages.append(Package('amsmath'))
        self.packages.append(Package('float'))
        self.packages.append(Package('booktabs'))
        self.packages.append(Package('geometry', options=['margin=1in']))
        self.packages.append(Package('caption'))
        self.packages.append(Package('subcaption'))

class ModelComparisonReport:
    def __init__(self, plots_dir, exp_id, model_colors, model_name_map):
        self.plots_dir = plots_dir
        self.exp_id = exp_id
        self.model_colors = model_colors
        self.model_name_map = model_name_map
        self.doc = CustomDocument()
        
    def generate_report(self, all_results, metrics):
        # Add title
        self.doc.preamble.append(NoEscape(r'\title{Model Comparison Analysis}'))
        self.doc.preamble.append(NoEscape(r'\author{Generated Report}'))
        self.doc.preamble.append(NoEscape(r'\date{\today}'))
        self.doc.append(NoEscape(r'\maketitle'))
        
        # Add configuration section
        self._add_configuration(all_results)
        
        # Add analysis section
        self._add_analysis(metrics)
        
        # Generate PDF
        output_path = os.path.join(self.plots_dir, f"report_{self.exp_id}")
        self.doc.generate_pdf(output_path, clean_tex=False)
        
    def _add_configuration(self, all_results):
        with self.doc.create(Section('Configuration Details')):
            config_columns = [
                "batch_size", "dataset_name", "exp_id", "max_entries",
                "max_new_tokens", "model_name", "n_beams", "n_repeats",
                "strategy", "temperature", "typo_intensity", "typo_type",
                "use_beam_search"
            ]
            
            self.doc.append(NoEscape(r'\begin{description}'))
            for col in config_columns:
                unique_values = all_results[f"config.{col}"].unique()
                values_str = ", ".join(map(str, unique_values))
                self.doc.append(NoEscape(f'\\item[{col}] {values_str}'))
            self.doc.append(NoEscape(r'\end{description}'))
    
    def _add_analysis(self, metrics):
        with self.doc.create(Section('Analysis Results')):
            for metric in metrics:
                self._add_metric_analysis(metric)
    
    def _add_metric_analysis(self, metric):
        # Add metric subsection
        with self.doc.create(Section(f'{metric} Analysis', numbering=False)):
            # Add formula
            if metric == "Accuracy":
                formula = r'\text{Accuracy} = \frac{\sum I(\text{true\_answer} \in \text{generated\_answer})}{|\text{dataset}|}'
                self.doc.append(NoEscape(r'\begin{equation}' + formula + r'\end{equation}'))
            elif metric == "AUCPR_sample":
                formula = r'\text{AUCPR} = \int_0^1 \frac{TP}{TP + FP} d(\frac{TP}{TP + FN})'
                self.doc.append(NoEscape(r'\begin{equation}' + formula + r'\end{equation}'))
            
            # Add plots
            self._add_plots_for_metric(metric)
    
    def _add_plots_for_metric(self, metric):
        # Add radar plots
        for intensity in [1, 2, 3]:
            plot_path = os.path.join(
                self.plots_dir, 
                f"radar_plot_{metric}_intensity_{intensity}.png"
            )
            
            if os.path.exists(plot_path):
                with self.doc.create(Figure(position='H')) as fig:
                    fig.add_image(plot_path, width=NoEscape(r'0.8\textwidth'))
                    fig.add_caption(f'Radar Plot - Intensity {intensity}')
        
        # Add box plot
        box_plot_path = os.path.join(self.plots_dir, f"box_plot_{metric}.png")
        if os.path.exists(box_plot_path):
            with self.doc.create(Figure(position='H')) as fig:
                fig.add_image(box_plot_path, width=NoEscape(r'0.8\textwidth'))
                fig.add_caption('Box Plot Distribution')
        
        # Add bar plots
        for intensity in [1, 2, 3]:
            plot_path = os.path.join(
                self.plots_dir, 
                f"bar_plot_{metric}_intensity_{intensity}.png"
            )
            
            if os.path.exists(plot_path):
                with self.doc.create(Figure(position='H')) as fig:
                    fig.add_image(plot_path, width=NoEscape(r'0.8\textwidth'))
                    fig.add_caption(f'Bar Plot - Intensity {intensity}')

def generate_report_with_pylatex(all_results, exp_id, model_colors, model_name_map):
    """Main function to generate the report"""
    # Create output directory
    plots_dir = f"plots/model_comparison_{exp_id}"
    os.makedirs(plots_dir, exist_ok=True)
    
    # Initialize report generator
    report = ModelComparisonReport(plots_dir, exp_id, model_colors, model_name_map)
    
    # Generate report
    try:
        report.generate_report(all_results, ["Accuracy", "AUCPR_sample"])
        print(f"PDF generated successfully in {plots_dir}")
    except Exception as e:
        print(f"Error generating PDF: {e}")

ModuleNotFoundError: No module named 'tectonic'

## 4. Debug HQQ

In [8]:
import pandas as pd
import numpy as np

# Parameters for filtering
metric = 'Accuracy'  # or 'AUCPR_sample'
model = 'Llama-3-8B-HQQ-mixed-local'  # adjust as needed

# Get the exact filtered DataFrame we're interested in
filter_condition = (all_results['config.typo_type'] == 'none') & (all_results['config.model_name'] == model)
filtered_df = all_results[filter_condition]

# Print information about the filtering
print(f"\nFiltering for:")
print(f"config.typo_type == 'none' AND config.model_name == '{model}'")

print(f"\nNumber of rows found: {len(filtered_df)}")

# Print the complete filtered DataFrame
print("\nComplete filtered DataFrame:")
print(filtered_df)

# Print the specific columns we're most interested in
columns_of_interest = ['config.typo_type', 'config.model_name', f'result.{metric}']
print(f"\nFiltered DataFrame (key columns only):")
print(filtered_df[columns_of_interest])

# Print the actual baseline value that would be used
if not filtered_df.empty:
    baseline_value = filtered_df[f'result.{metric}'].iloc[0]
    print(f"\nBaseline value that would be used: {baseline_value}")
    
    # Additional validation
    if len(filtered_df) > 1:
        print("\nWARNING: Multiple rows found! All values:")
        print(filtered_df[f'result.{metric}'].values)
else:
    print(f"\nWARNING: No data found for model '{model}' with typo_type 'none'")

# Print unique values in key columns to help with debugging
print("\nUnique values in key columns:")
print("\nUnique typo_types:")
print(all_results['config.typo_type'].unique())
print("\nUnique model_names:")
print(all_results['config.model_name'].unique())


Filtering for:
config.typo_type == 'none' AND config.model_name == 'Llama-3-8B-HQQ-mixed-local'

Number of rows found: 60

Complete filtered DataFrame:
       _id  config.overwrite    config.db_collection  config.batch_size  \
3550  3603              3603  llama-pert-awq-bnb-hqq                 32   
3590  3661              3661  llama-pert-awq-bnb-hqq                 32   
3591  3662              3662  llama-pert-awq-bnb-hqq                 32   
3592  3663              3663  llama-pert-awq-bnb-hqq                 32   
3650  3721              3721  llama-pert-awq-bnb-hqq                 32   
3651  3722              3722  llama-pert-awq-bnb-hqq                 32   
3652  3723              3723  llama-pert-awq-bnb-hqq                 32   
3710  3781              3781  llama-pert-awq-bnb-hqq                 32   
3711  3782              3782  llama-pert-awq-bnb-hqq                 32   
3712  3783              3783  llama-pert-awq-bnb-hqq                 32   
3770  3841            

In [10]:
all_results["config.model_name"].unique()

array(['Llama-3-8B', 'Llama-3-8B-AWQ-4bit-local',
       'Llama-3-8B-BNB-4bit-local', 'Llama-3-8B-HQQ-mixed-local'],
      dtype=object)

In [9]:
import pandas as pd
import numpy as np

# Models to compare
hqq_model = 'Llama-3-8B-HQQ-mixed-local'
base_model = 'Llama-3-8B'
metric = 'Accuracy'

# Get filtered DataFrames for both models
hqq_filter = (all_results['config.typo_type'] == 'none') & (all_results['config.model_name'] == hqq_model)
base_filter = (all_results['config.typo_type'] == 'none') & (all_results['config.model_name'] == base_model)

hqq_df = all_results[hqq_filter]
base_df = all_results[base_filter]

# Print side-by-side comparison
print("\n=== Model Comparison (Baseline Accuracy) ===")
print("-" * 50)
print(f"HQQ Model: {hqq_model}")
print(f"Base Model: {base_model}")
print("-" * 50)

if not hqq_df.empty and not base_df.empty:
    hqq_accuracy = hqq_df[f'result.{metric}'].iloc[0]
    base_accuracy = base_df[f'result.{metric}'].iloc[0]
    
    print(f"\nAccuracy Values:")
    print(f"{'Model':<30} {'Accuracy':<10}")
    print("-" * 40)
    print(f"{hqq_model:<30} {hqq_accuracy:.4f}")
    print(f"{base_model:<30} {base_accuracy:.4f}")
    
    # Calculate difference
    diff = hqq_accuracy - base_accuracy
    print(f"\nDifference (HQQ - Base): {diff:.4f}")
    print(f"Relative Change: {(diff/base_accuracy)*100:.2f}%")
    
    # Print warning if multiple rows found
    if len(hqq_df) > 1:
        print(f"\nWARNING: Multiple rows found for HQQ model! All values:")
        print(hqq_df[f'result.{metric}'].values)
    if len(base_df) > 1:
        print(f"\nWARNING: Multiple rows found for Base model! All values:")
        print(base_df[f'result.{metric}'].values)
else:
    if hqq_df.empty:
        print(f"No data found for HQQ model with typo_type 'none'")
    if base_df.empty:
        print(f"No data found for Base model with typo_type 'none'")

# Print full filtered DataFrames for verification
print("\n=== Full Filtered DataFrames ===")
print("\nHQQ Model DataFrame:")
print(hqq_df[['config.typo_type', 'config.model_name', f'result.{metric}']])
print("\nBase Model DataFrame:")
print(base_df[['config.typo_type', 'config.model_name', f'result.{metric}']])


=== Model Comparison (Baseline Accuracy) ===
--------------------------------------------------
HQQ Model: Llama-3-8B-HQQ-mixed-local
Base Model: Llama-3-8B
--------------------------------------------------

Accuracy Values:
Model                          Accuracy  
----------------------------------------
Llama-3-8B-HQQ-mixed-local     0.3448
Llama-3-8B                     0.3966

Difference (HQQ - Base): -0.0517
Relative Change: -13.04%

[0.34482759 0.87819857 0.87819857 0.87819857 0.4386423  0.4386423
 0.4386423  0.7026087  0.7026087  0.7026087  0.71794872 0.72649573
 0.72649573 0.80701754 0.80701754 0.80701754 0.67321613 0.67218201
 0.67218201 0.6827957  0.6827957  0.6827957  0.92464358 0.92464358
 0.92464358 0.90709459 0.90709459 0.90709459 0.25529661 0.25529661
 0.25529661 0.36621196 0.36621196 0.36621196 0.29370629 0.3030303
 0.3030303  0.71221532 0.71221532 0.71221532 0.6903024  0.6903024
 0.6903024  0.77128205 0.77128205 0.77128205 0.64953271 0.64953271
 0.64953271 0.7049689

## 5. Remove Duplicates

In [5]:
import pandas as pd

def inspect_and_remove_duplicates(df):
    # Define the columns used for pivoting
    pivot_columns = ['config.typo_type', 'config.typo_intensity']
    
    # Find duplicates in pivot columns
    duplicate_mask = df.duplicated(subset=pivot_columns, keep=False)
    duplicates = df[duplicate_mask]
    
    if duplicates.empty:
        print("No duplicates found in pivot columns.")
        return df
    
    print("Duplicate entries found in pivot columns:")
    print(duplicates[pivot_columns])
    
    print("\nFull rows for duplicate entries:")
    print(duplicates)
    
    # Ask user how to handle duplicates
    print("\nHow would you like to handle these duplicates?")
    print("1: Keep first occurrence")
    print("2: Keep last occurrence")
    print("3: Remove all duplicates")
    print("4: Do nothing (keep all)")
    
    choice = input("Enter your choice (1-4): ")
    
    if choice == '1':
        df_cleaned = df.drop_duplicates(subset=pivot_columns, keep='first')
        print(f"Removed {len(df) - len(df_cleaned)} duplicate rows.")
    elif choice == '2':
        df_cleaned = df.drop_duplicates(subset=pivot_columns, keep='last')
        print(f"Removed {len(df) - len(df_cleaned)} duplicate rows.")
    elif choice == '3':
        df_cleaned = df.drop_duplicates(subset=pivot_columns, keep=False)
        print(f"Removed {len(df) - len(df_cleaned)} duplicate rows.")
    elif choice == '4':
        df_cleaned = df
        print("No rows removed.")
    else:
        print("Invalid choice. No rows removed.")
        df_cleaned = df
    
    return df_cleaned

# Assuming your dataframe is named 'all_results'
all_results_llama = inspect_and_remove_duplicates(all_results_llama)

# You can now use all_results_cleaned for further processing

Duplicate entries found in pivot columns:
           config.typo_type  config.typo_intensity
42  word_phrase_translation                      1
43  word_phrase_translation                      2
59  word_phrase_translation                      1
60  word_phrase_translation                      2

Full rows for duplicate entries:
    _id  config.overwrite config.db_collection config.dataset_name  \
42   43                43      llama-typo-eval                 P17   
43   44                44      llama-typo-eval                 P17   
59   61                61      llama-typo-eval                 P17   
60   62                62      llama-typo-eval                 P17   

   config.device    config.exp_id config.max_entries  config.max_new_tokens  \
42          cuda  typo-test-10-01               None                     25   
43          cuda  typo-test-10-01               None                     25   
59          cuda  typo-test-10-01               None                     25   
60